In [7]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# Define dataset paths
root_dir = 'C:\\Users\\duaas\\Downloads\\QC'
intact_side_dir = os.path.join(root_dir, 'intact', 'side')
intact_top_dir = os.path.join(root_dir, 'intact', 'top')
damaged_side_dir = os.path.join(root_dir, 'damaged', 'side')
damaged_top_dir = os.path.join(root_dir, 'damaged', 'top')

# Define data augmentation parameters
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Load training data
train_generator = train_datagen.flow_from_directory(
    root_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

# Load pre-trained DenseNet121 model
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(train_generator, epochs=25)

# Save the model
model.save('medicine_package_classification_model_densenet.h5')


Found 400 images belonging to 2 classes.
Epoch 1/25
13/13 [==============================] - 54s 3s/step - loss: 0.8014 - accuracy: 0.4975
Epoch 2/25
13/13 [==============================] - 34s 3s/step - loss: 0.7520 - accuracy: 0.5175
Epoch 3/25
13/13 [==============================] - 33s 2s/step - loss: 0.7449 - accuracy: 0.5300
Epoch 4/25
13/13 [==============================] - 31s 2s/step - loss: 0.7014 - accuracy: 0.5650
Epoch 5/25
13/13 [==============================] - 32s 2s/step - loss: 0.7101 - accuracy: 0.5550
Epoch 6/25
13/13 [==============================] - 32s 2s/step - loss: 0.6474 - accuracy: 0.5975
Epoch 7/25
13/13 [==============================] - 32s 2s/step - loss: 0.6373 - accuracy: 0.6100
Epoch 8/25
13/13 [==============================] - 31s 2s/step - loss: 0.6371 - accuracy: 0.6050
Epoch 9/25
13/13 [==============================] - 32s 2s/step - loss: 0.6180 - accuracy: 0.6500
Epoch 10/25
13/13 [==============================] - 31s 2s/step - loss: 0.60

In [8]:
# Evaluate the model on test data
test_loss, test_accuracy = model.evaluate(train_generator)
print("Test Accuracy:", test_accuracy)


13/13 [==============================] - 15s 1s/step - loss: 0.5505 - accuracy: 0.7000
Test Accuracy: 0.699999988079071
